[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zjelveh/zjelveh.github.io/blob/master/files/ccjs418e_fall2026/10_complete_analysis.ipynb)

**Save your own copy.**
1. Click File > Save a copy in Drive.
2. Rename it.
3. Work in your copy.

---

# 10. From Question to Complete Analysis: Yankees Games and Bronx Noise
## CCJS 418E: Coding for Criminology — Fall 2026

On the first day of class we asked a question about Yankees home games and 311 noise complaints. Since then you have learned every tool a full answer needs: filtering, grouping, dates, charts, and merging. This notebook puts them together, from the question to a claim.

> Do Yankees home games coincide with more 311 noise complaints in the Bronx?

We will keep returning to three questions:

1. What do you want to know?
2. Compared to what?
3. What can you honestly say?

One cell begins with `# Expected ValueError`. It fails on purpose. Read the error, then run the next cell.

In [ ]:
# Show the result of every line in a cell, so one cell can display several outputs
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Print NumPy numbers as plain values, such as 3 instead of np.int64(3)
np.set_printoptions(legacy="1.25")
# Draw charts on a white background with light grid lines
sns.set_style('whitegrid')
# Make every chart 9 inches wide and 4 inches tall
plt.rcParams['figure.figsize'] = (9, 4)

## Setup: Two Tables

**`complaints`: one row represents one 311 noise complaint.** The table holds 2023 complaints of three noise types (residential, street/sidewalk, and commercial) from the Bronx and Brooklyn.

| Column | Meaning |
|---|---|
| `created_date` | Date and time the complaint was made |
| `borough` | `BRONX` or `BROOKLYN` |
| `complaint_type` | `Noise - Residential`, `Noise - Street/Sidewalk`, or `Noise - Commercial` |
| `descriptor` | More detail, such as `Loud Music/Party` |

**`games`: one row represents one Yankees home game in the 2023 regular season.** Yankee Stadium is in the Bronx. The schedule was checked against MLB's official records.

| Column | Meaning |
|---|---|
| `game_date` | Date of the game |
| `opponent` | Visiting team |
| `result` | `W` if the Yankees won, `L` if they lost |

In [ ]:
complaints_url = 'https://raw.githubusercontent.com/zjelveh/zjelveh.github.io/refs/heads/master/files/ccjs418e_fall2026/nyc_311_noise_sample.csv'
games_url = 'https://raw.githubusercontent.com/zjelveh/zjelveh.github.io/refs/heads/master/files/ccjs418e_fall2026/yankees_home_games_2023.csv'

complaints = pd.read_csv(filepath_or_buffer=complaints_url)
games = pd.read_csv(filepath_or_buffer=games_url)

## Part 1: The Question and the Plan

**What do you want to know?** Whether the Bronx records more noise complaints on days the Yankees play at home.

**Compared to what?** Days without a home game. Choosing *which* days is the main decision in this analysis. We will build three benchmarks, each closer to a fair comparison, and watch the answer change.

**What can you honestly say?** We will decide at the end, from the strongest comparison we build.

The plan follows the steps from the first lecture:

1. Load the two tables.
2. Count complaints per day in the Bronx.
3. Mark the days with a home game.
4. Compare game days with a benchmark, then improve the benchmark.

This week's reading from The Trace compares D.C. shootings before and after the National Guard deployment and looks at the trend that came before it. A pattern already under way can make a before-and-after difference look larger or smaller. The baseball season plays that role here: noise complaints rise in warm months, and baseball is played in warm months.

## Part 2: Look at the Tables

`complaints` has one row per complaint. Display its size and first rows.

In [ ]:
complaints.shape
complaints.head(n=3)

The table covers two boroughs and three complaint types.

In [ ]:
complaints['borough'].value_counts()
complaints['complaint_type'].value_counts()

`games` has one row per game. Display its size, its first rows, and its first and last dates.

In [ ]:
games.shape
games.head(n=3)
games['game_date'].min()
games['game_date'].max()

## Part 3: Count Complaints per Day in the Bronx

The question compares days, so we need a table with one row per day. Start by converting `created_date` to datetime values, as in the datetime notebook.

In [ ]:
complaints['created_date'] = pd.to_datetime(arg=complaints['created_date'])

complaints['created_date'].head(n=3)

Each value includes a time of day. To count by day, every complaint needs a column holding only its date.

In the plotting-with-dates notebook, `.dt.to_period(freq='W').dt.to_timestamp()` labeled each report with the first day of its week. With `freq='D'`, the same pattern labels each complaint with its day: the time becomes midnight, so all complaints from one day share one value.

In [ ]:
complaints['date'] = complaints['created_date'].dt.to_period(freq='D').dt.to_timestamp()

complaints[['created_date', 'date']].head(n=3)

Keep the Bronx complaints, then count the complaints on each date. `.reset_index(name='complaints')` turns the counts into a table with a `complaints` column, as in the visualization notebook.

In [ ]:
bronx = complaints[complaints['borough'] == 'BRONX']
daily = bronx.groupby(by=['date']).size().reset_index(name='complaints')

daily.head(n=3)
daily.shape

Check the table before using it. 2023 had 365 days, and the daily counts should add up to the number of Bronx complaints.

In [ ]:
len(bronx)
daily['complaints'].sum()

## Part 4: Mark the Game Days

To compare game days with other days, each row of `daily` needs to say whether the Yankees played at home that day. `games` has that information, so we merge it in.

A merge needs a column with the same name in both tables. `daily` calls its date column `date`, so copy `game_date` into a new column named `date`.

In [ ]:
# Expected ValueError: merge a date column with a text column
games['date'] = games['game_date']
pd.merge(left=daily, right=games, on='date', how='left')

The error says one `date` column holds `datetime64[ns]` values and the other holds `object` values. `daily['date']` holds dates; `games['date']` holds the text copied from `game_date`. Pandas matches values only when both columns hold the same type, so convert the games column to dates and merge again.

This error catches a mismatch in type, and a mismatch in detail produces no error at all: had we skipped `.dt.to_period(freq='D')` in Part 3, a complaint made at 9:14 p.m. would never equal a game date stored as midnight, and only 1 of the 122,900 Bronx complaints would match a game. The 365-row check in Part 3 is what catches that mistake.

In [ ]:
games['date'] = pd.to_datetime(arg=games['game_date'])
with_games = pd.merge(left=daily, right=games, on='date', how='left')

with_games.shape
daily.shape

The merge returned 366 rows, one more than `daily`. A left merge keeps every row of the left table, and it repeats a left row when that row matches more than one row on the right. Count how often each date appears.

In [ ]:
with_games['date'].value_counts().head(n=3)

June 8 appears twice. Display those rows.

In [ ]:
with_games[with_games['date'] == pd.to_datetime(arg='2023-06-08')]

The Yankees played the Chicago White Sox twice on June 8, a doubleheader: they lost the first game and won the second. `games` has one row per game, so it has 81 rows but 80 different dates.

In [ ]:
len(games)
games['date'].nunique()

Our question compares **days**, so the comparison table needs one row per day. `.isin()` checks each date in `daily` against the game dates and returns one `True` or `False` per day, so `daily` keeps its 365 rows. Part 8 asks a question about **games** and merges in the other direction.

`.sum()` counts each `True` as 1, so it gives the number of game days.

In [ ]:
daily['game_day'] = daily['date'].isin(values=games['date'])

daily['game_day'].sum()
daily.shape

Add the day of the week and the month. Adding them to `daily` now means every table we filter from it later includes them.

In [ ]:
daily['weekday'] = daily['date'].dt.day_name()
daily['month'] = daily['date'].dt.month

daily.head(n=3)

## Part 5: First Benchmark — All Other Days

Average the daily counts separately for game days and for all other days.

In [ ]:
daily.groupby(by=['game_day'])['complaints'].mean().round(decimals=1)

Game days averaged 429.0 complaints and all other days 310.8, a gap of 118.2.

In the first lecture these numbers were 430 and 309. That version of the games file listed 83 dates, including three games the Yankees played at Dodger Stadium in Los Angeles (June 2–4). The file in this notebook was checked against MLB's official schedule. Checking what each row represents applies to every file, including the instructor's.

Which days make up "all other days"? Plot every day of 2023, color the game days, and mark the first and last home games.

In [ ]:
first_game = games['date'].min()
last_game = games['date'].max()

sns.scatterplot(data=daily, x='date', y='complaints', hue='game_day')
plt.axvline(x=first_game, color='gray', linestyle='--')
plt.axvline(x=last_game, color='gray', linestyle='--')
plt.title(label='Bronx Noise Complaints per Day, 2023')
plt.ylabel(ylabel='Complaints')
plt.tick_params(axis='x', labelrotation=45)

Every game falls between the dashed lines, from March 30 to September 25. Complaints run higher in those months than in winter, with or without a game. Of the 285 comparison days, 185 fall outside the season, so this benchmark mixes the season with whatever the games add.

## Part 6: Second Benchmark — Non-Game Days During the Season

Keep the days from the first home game through the last with `.between()`, from the datetime notebook.

In [ ]:
season = daily[daily['date'].between(left=first_game, right=last_game)]

len(season)
season['game_day'].value_counts()

In [ ]:
season.groupby(by=['game_day'])['complaints'].mean().round(decimals=1)

Within the season, non-game days averaged 400.8 complaints. The gap fell from 118.2 to 28.2. The game days stayed the same; only the benchmark changed.

## Part 7: Third Benchmark — The Same Days of the Week

Noise may differ by day of the week. Compare game days and non-game days within each weekday.

In [ ]:
weekday_avg = season.groupby(by=['weekday', 'game_day'])['complaints'].mean().round(decimals=1).reset_index(name='avg_complaints')
week_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

sns.barplot(data=weekday_avg, x='weekday', y='avg_complaints', hue='game_day', order=week_order)
plt.title(label='Average Bronx Noise Complaints by Weekday, 2023 Season')
plt.ylabel(ylabel='Complaints per day')

Saturdays and Sundays average about 600–740 complaints, Monday through Thursday about 250–350, and Fridays about 350–400.

Each pair of bars is a fair comparison on its own: game days and non-game days on the same weekday. Game days run higher on five of the seven weekdays and lower on Mondays and Wednesdays, by amounts much smaller than the differences between weekdays. The chart gives seven answers, and the main question needs one. Combining them means deciding how much each weekday counts.

> **Going further.** Part 6's in-season comparison is a solid stopping point, and it is enough for a final project. The rest of Part 7 builds a more careful benchmark that combines the seven weekday comparisons, a step to take when you want to be especially careful. Run its cells either way: Part 8 uses their results.

The Part 6 benchmark (400.8) already combines the weekdays, counting each one by how often it appears among the non-game days. If game days fall on different weekdays than non-game days do, that benchmark compares two different calendars. Compare the weekday mix of the two groups.

In [ ]:
season[season['game_day'] == True]['weekday'].value_counts(normalize=True).round(decimals=3)
season[season['game_day'] == False]['weekday'].value_counts(normalize=True).round(decimals=3)

Mondays, a common day off in baseball schedules, make up 7.5% of game days and 20.0% of non-game days. Saturdays and Sundays make up 31.2% of game days and 27.0% of non-game days. The Part 6 benchmark counts Mondays far more heavily than the game days do.

A fairer benchmark compares each game day with non-game days on the **same weekday**, counting each weekday as often as the game days do. It takes two steps:

1. Average the non-game days for each weekday, giving seven numbers: the non-game bars in the chart.
2. Look up the matching average for each game day with a merge, the way the merging notebook looked up each borough's county. A weekday with 13 game days gets looked up 13 times; Monday, with 6 game days, gets looked up 6 times.

In [ ]:
non_game_weekday = season[season['game_day'] == False].groupby(by=['weekday'])['complaints'].mean().reset_index(name='non_game_average')

non_game_weekday.round(decimals=1)

In [ ]:
game_days = pd.merge(left=season[season['game_day'] == True], right=non_game_weekday, on='weekday', how='left')

game_days.shape
game_days[['date', 'weekday', 'complaints', 'non_game_average']].head(n=3).round(decimals=1)

Each game day now carries the average for non-game days on its weekday. Averaging that column gives the benchmark: the number of complaints game days would average if each matched a typical non-game day on the same weekday.

In [ ]:
game_days['complaints'].mean().round(decimals=1)
game_days['non_game_average'].mean().round(decimals=1)

Matched by weekday, the benchmark is 408.8 and the gap is 20.2 complaints per game day. This number combines the chart's seven comparisons, counting each weekday once for every game played on it.

An average can hide how the days differ. Calculate each game day's difference from its benchmark and count the days above it.

In [ ]:
game_days['difference'] = game_days['complaints'] - game_days['non_game_average']

(game_days['difference'] > 0).sum()
len(game_days)

Only 34 of the 80 game days (42.5%) came in above their benchmark. The positive average comes from a smaller number of days with large increases. Sort to see them.

In [ ]:
game_days[['date', 'weekday', 'complaints', 'non_game_average', 'difference']].sort_values(by='difference', ascending=False).head(n=5).round(decimals=1)

The largest gap of the season fell on July 4, Independence Day, when fireworks and parties bring noise complaints whether or not there is a game. July 5, the day after, also ranks near the top. Check how much one holiday moves the average: drop July 4 and recompute.

In [ ]:
without_july_4 = game_days[game_days['date'] != pd.to_datetime(arg='2023-07-04')]

len(without_july_4)
without_july_4['difference'].mean().round(decimals=1)

Without July 4, the gap falls from 20.2 to 12.9 complaints per game day. One day out of 80 accounts for more than a third of the gap.

## Part 8: Wins and Losses

Did complaints differ after wins and after losses? This question is about **games**, so start from `games`, one row per game, and merge in each day's numbers. June 8 then appears twice, once for each game.

In [ ]:
by_game = pd.merge(left=games, right=game_days, on='date', how='left')

by_game.shape
by_game['difference'].isna().sum()

In [ ]:
by_game['result'].value_counts()
by_game.groupby(by=['result'])['difference'].mean().round(decimals=1)

Loss games came in 30.9 complaints above their benchmark and win games 9.7 above. Read these numbers with care:

- The groups are small: 39 losses and 42 wins. A few unusual days can move either average.
- July 4, the largest gap of the season, was a win.
- June 8's complaints count once in each group, because both games share one day.
- A daily count includes complaints made before the game started, which no result could affect.

## Part 9: Answer the Main Question

| Benchmark | Days | Average | Game-day gap (429.0) |
|---|---:|---:|---:|
| All other 2023 days | 285 | 310.8 | +118.2 |
| Non-game days in the season | 100 | 400.8 | +28.2 |
| Non-game days in the season, matched by weekday (going further) | 100 | 408.8 | +20.2 |

**What do we want to know?** Whether the Bronx records more noise complaints on days the Yankees play at home.

**Compared to what?** Non-game days during the same season. Matching those days by weekday, as in the second half of Part 7, is a more careful version of the same comparison.

**What can we honestly say?** In 2023, the Bronx averaged about 28 more noise complaints per day on Yankees home-game days (429.0) than on non-game days during the season (400.8), about 7% more. Matching each game day with non-game days on the same weekday narrows the gap to about 20, and to about 13 without July 4, a holiday that fell on a game day. The gap shrank from 118 as the benchmark improved, and fewer than half of the game days came in above their weekday-matched benchmark. These numbers describe an association between game days and complaints in one borough and one season.

Next week we strengthen the comparison. Brooklyn, far from the stadium, shows what happened on the same dates elsewhere in the city, and heating complaints, which a baseball game should leave unchanged, test whether the method finds gaps where none should exist.

---

## Hands-On Exercises

Each exercise has two parts. Part A repeats an example with one change. Part B changes the comparison. Exercise 1 practices the main comparisons from Parts 5 and 6. Exercises 2 and 3 go further, using the matched benchmark from the second half of Part 7, and each ends with a short written answer.

### Exercise 1: Street and Sidewalk Noise

Crowds arriving at and leaving a stadium would most likely show up as street noise.

**Part A:** Build a daily table of Bronx complaints whose `complaint_type` is `'Noise - Street/Sidewalk'`. Name it `street_daily` and add `game_day` and `weekday` columns. Check that it has 365 rows, then compare the average on game days with the average on all other days.

**Part B:** Keep the days from `first_game` through `last_game` in a table named `street_season`, and compare game days with non-game days.

In [ ]:
# Part A: Build street_daily, add game_day and weekday, and compare game days with all other days.

In [ ]:
# Part B: Keep the season days in street_season and compare game days with non-game days.

### Exercise 2 (Going Further): Which Complaint Type Carries the Gap?

**Part A:** Using `street_season`, build the weekday-matched benchmark for street and sidewalk noise, as in Part 7. Display the game-day average, the benchmark, and the gap.

**Part B:** Repeat Exercise 1 and Part A for `'Noise - Residential'`. Then answer in 2–3 sentences: which complaint type accounts for most of the 20.2-complaint gap for all noise?

In [ ]:
# Part A: Build the weekday-matched benchmark for street and sidewalk noise.

In [ ]:
# Part B: Repeat the daily table, season filter, and weekday-matched benchmark for residential noise.

**Your answer:**

### Exercise 3 (Going Further): Match by Month Instead

Part 7 matched each game day with non-game days on the same weekday. Matching on the month is another choice.

**Part A:** Average the complaints in `season` by `month` and `game_day`, and draw a bar chart with `hue='game_day'`. In which months did game days average more complaints than non-game days?

**Part B:** Build a month-matched benchmark: average the non-game days in `season` by `month`, merge those averages onto the game days, and average them. Then answer in 2–3 sentences: why does matching on month leave the benchmark near the in-season average (400.8), while matching on weekday raises it to 408.8?

In [ ]:
# Part A: Average season complaints by month and game_day, then draw the bar chart.

In [ ]:
# Part B: Build the month-matched benchmark with a merge on month.

**Your answer:**

## Reference: From Records to a Matched Comparison

```python
# Label each timestamp with its day
complaints['date'] = complaints['created_date'].dt.to_period(freq='D').dt.to_timestamp()

# Count records per day
daily = bronx.groupby(by=['date']).size().reset_index(name='complaints')

# Mark the days that appear in another table, one True or False per day
daily['game_day'] = daily['date'].isin(values=games['date'])

# Keep a date range
season = daily[daily['date'].between(left=first_game, right=last_game)]

# Matched benchmark: average the comparison days by weekday, then look up each game day's match
non_game_weekday = season[season['game_day'] == False].groupby(by=['weekday'])['complaints'].mean().reset_index(name='non_game_average')
game_days = pd.merge(left=season[season['game_day'] == True], right=non_game_weekday, on='weekday', how='left')
game_days['non_game_average'].mean()
```

| What you see | Cause | Fix |
|---|---|---|
| `ValueError` about `datetime64[ns]` and `object` columns | One merge column holds dates and the other holds text | Convert the text column with `pd.to_datetime()` |
| More rows after a left merge than before | Some left rows match more than one right row | Find the repeated values with `.value_counts()` and use the table whose rows match your question |
| A benchmark that changes when you match on another column | The two groups differ on that column | Report the matched comparison and say what it matches on |